# ARC26 Vanilla V2 q9/24 plus bounded TRM-64

The exact 32.08 Vanilla V2 production path, with inference batching changed from
4 to 6. After Qwen finishes, the released TRM v18 recipe uses 64 augmentations
and trains until the shared 11h25 deadline, then evaluates every task. Final
attempts are Qwen rank 1 plus TRM attempt 1, falling back to Qwen rank 2 only
when the TRM grid is invalid or duplicates Qwen rank 1.


In [ ]:
import time

NOTEBOOK_START_TIME = time.time()
TRM_AUGMENTATIONS = 64
TRM_TRAIN_STOP_HOURS = 11 + 25 / 60
FINAL_TARGET_HOURS = 11 + 50 / 60
print("notebook_start_time =", NOTEBOOK_START_TIME)


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "submit_competition"  # validation | submit_competition

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

VALIDATION_KEYS = None
NPROCS = 4
DFS_PROB_THRESHOLD = 0.2
UNSLOTH_MULTITOKEN_REPEAT_LEN = 9
EVAL_COLOR_PERMUTATIONS = 3
EVAL_BATCH_SIZE = 6
SELECTION_ALGORITHM = "score_kgmon"
PROFILE_TIMINGS = True

VALIDATION_END_TIME_HOURS = 2.5
SUBMIT_COMPETITION_END_TIME_HOURS = 11 + 50 / 60
RESET_RUN_ARTIFACTS = True

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc2026_run_vanilla_v2_q9_24"
WORK_CODE_DIR = "/kaggle/working/arc2026_run_vanilla_v2_q9_24/ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = "/kaggle/working/vanilla_v2_q9_24_stack"


In [ ]:
import os
from pathlib import Path


def _truthy_env(name: str) -> bool:
    value = os.getenv(name)
    if value is None:
        return False
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE_RERUN = _truthy_env("KAGGLE_IS_COMPETITION_RERUN")
assert MODE in {"validation", "submit_competition"}
EFFECTIVE_MODE = "submit_competition" if IS_KAGGLE_RERUN else MODE

EVAL_CHALLENGES = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
EVAL_SOLUTIONS = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
TEST_CHALLENGES = f"{COMP_ROOT}/arc-agi_test_challenges.json"

if IS_KAGGLE_RERUN:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_submit"
    SUBMISSION_PATH = "/kaggle/working/qwen_submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = SUBMIT_COMPETITION_END_TIME_HOURS
    RUN_INFERENCE = True
elif MODE == "validation":
    TEST_PATH = EVAL_CHALLENGES
    SOLUTION_PATH = EVAL_SOLUTIONS
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_validation"
    SUBMISSION_PATH = "/kaggle/working/validation_submission_unsloth_q9.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = VALIDATION_KEYS
    END_TIME_HOURS = VALIDATION_END_TIME_HOURS
    RUN_INFERENCE = True
else:
    TEST_PATH = TEST_CHALLENGES
    SOLUTION_PATH = None
    OUTPUT_DIR = "/kaggle/working/inference_outputs_vanilla_v2_q9_24_shortcut"
    SUBMISSION_PATH = "/kaggle/working/qwen_submission.json"
    LIMIT_KEYS = None
    SELECTED_KEYS = None
    END_TIME_HOURS = 0.0
    RUN_INFERENCE = False

print("mode_requested =", MODE)
print("is_kaggle_rerun =", IS_KAGGLE_RERUN)
print("effective_mode =", EFFECTIVE_MODE)
print("test_path =", TEST_PATH)
print("output_dir =", OUTPUT_DIR)
print("submission_path =", SUBMISSION_PATH)
print("selected_keys =", SELECTED_KEYS)
print("end_time_hours =", END_TIME_HOURS)
print("run_inference =", RUN_INFERENCE)


In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(MODEL_PATH).exists(), f"Missing model path: {MODEL_PATH}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir("/kaggle/working")
print("setup cwd =", os.getcwd())

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT]:
        shutil.rmtree(path, ignore_errors=True)
    try:
        Path(SUBMISSION_PATH).unlink()
    except FileNotFoundError:
        pass
    for path in Path("/kaggle/working").glob("worker_train_*"):
        if path.is_file():
            path.unlink()

for module_name in ["unsloth", "transformers", "torch"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

src = Path(CODE_DATASET_ROOT)
dst = Path(WORK_NOTEBOOK_ROOT)
shutil.copytree(src, dst)

required_files = [
    "starter.py",
    "arc_solver.py",
    "arc_search_multitoken.py",
    "patch_unsloth_qwen3_multitoken.py",
]
for name in required_files:
    assert Path(WORK_CODE_DIR, name).is_file(), (
        f"arc2026 is stale: missing {name}"
    )

starter_source = Path(WORK_CODE_DIR, "starter.py").read_text()
solver_source = Path(WORK_CODE_DIR, "arc_solver.py").read_text()
assert "--use-unsloth-multitoken-dfs" in starter_source
assert "--eval-color-permutations" in starter_source
assert "UNSLOTH_COMPILE_LOCATION" in starter_source, "arc2026 starter.py lacks the worker import-race guard"
assert '"embed_tokens"' in solver_source and '"lm_head"' in solver_source
assert "inference_turbo_dfs_multitoken" in solver_source
print("arc2026 multi-token production preflight passed")
print("work_code_dir =", WORK_CODE_DIR)


In [ ]:
spec = importlib.util.find_spec("unsloth")
assert spec is not None and spec.submodule_search_locations
mounted_unsloth = Path(next(iter(spec.submodule_search_locations)))
qwen_source = (mounted_unsloth / "models" / "qwen3.py").read_text()
assert "A = flash_attn_func(Qnn, Knn, Vnn)" in qwen_source

writable_parent = Path(WRITABLE_UNSLOTH_PARENT)
writable_unsloth = writable_parent / "unsloth"
shutil.copytree(mounted_unsloth, writable_unsloth)

sys.path.insert(0, WORK_CODE_DIR)
from patch_unsloth_qwen3_multitoken import PATCH_MARKER, patch_unsloth

changed = patch_unsloth(writable_unsloth)
assert PATCH_MARKER in (writable_unsloth / "models" / "qwen3.py").read_text()
print("writable_unsloth =", writable_unsloth)
print("patched =", [str(path) for path in changed])

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(writable_parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


In [ ]:
import json
import os
import subprocess
import sys
import time

if RUN_INFERENCE:
    cmd = [
        sys.executable,
        "starter.py",
        "--test-path", TEST_PATH,
        "--model-path", MODEL_PATH,
        "--output-dir", OUTPUT_DIR,
        "--nprocs", str(NPROCS),
        "--use-unsloth-multitoken-dfs",
        "--unsloth-multitoken-repeat-len", str(UNSLOTH_MULTITOKEN_REPEAT_LEN),
        "--dfs-prob-threshold", str(DFS_PROB_THRESHOLD),
        "--eval-color-permutations", str(EVAL_COLOR_PERMUTATIONS),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--end-time", str(NOTEBOOK_START_TIME + TRM_TRAIN_STOP_HOURS * 3600),
    ]
    if PROFILE_TIMINGS:
        cmd.append("--profile-timings")
    if SELECTED_KEYS is not None:
        cmd.extend(["--keys-json", json.dumps(SELECTED_KEYS)])

    print("running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=WORK_CODE_DIR, env=RUN_ENV, check=True)
else:
    print("save-version shortcut: full inference runs only during the competition rerun")


In [ ]:
import json
import os
import sys
from pathlib import Path

if WORK_CODE_DIR not in sys.path:
    sys.path.insert(0, WORK_CODE_DIR)

from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon, score_full_probmul_3

SELECTION_ALGORITHMS = {
    "score_kgmon": score_kgmon,
    "score_full_probmul_3": score_full_probmul_3,
}


def _decoded_basekeys(output_dir):
    p = Path(output_dir)
    if not p.exists():
        return []
    return sorted({x.name.split(".")[0] for x in p.iterdir() if x.is_file()})


data = ArcDataset.from_file(TEST_PATH)
if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    data = data.load_replies(SOLUTION_PATH)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
if Path(OUTPUT_DIR).exists() and any(Path(OUTPUT_DIR).iterdir()):
    decoder.load_decoded_results(OUTPUT_DIR)

selection_algorithm = SELECTION_ALGORITHMS[SELECTION_ALGORITHM]
submission = data.get_submission(decoder.run_selection_algo(selection_algorithm) if decoder.decoded_results else None)

with open(SUBMISSION_PATH, "w") as f:
    json.dump(submission, f)

print("decoded_output_keys =", len(decoder.decoded_results))
print("decoded_basekeys =", _decoded_basekeys(OUTPUT_DIR)[:20])
print("submission_path =", SUBMISSION_PATH)
print("submission_tasks =", len(submission))
print("submission_exists =", Path(SUBMISSION_PATH).exists())

if EFFECTIVE_MODE == "validation" and SOLUTION_PATH is not None:
    if decoder.decoded_results:
        decoder.benchmark_selection_algos()
    with open(SUBMISSION_PATH) as f:
        reload_submission = json.load(f)
    print("validation_score =", data.validate_submission(reload_submission))
else:
    preview_keys = list(submission)[:5]
    print("preview_keys =", preview_keys)


In [ ]:
import shutil
from pathlib import Path

QWEN_SUBMISSION_PATH = Path("/kaggle/working/qwen_submission.json")
assert QWEN_SUBMISSION_PATH.is_file(), QWEN_SUBMISSION_PATH
for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT]:
    shutil.rmtree(path, ignore_errors=True)
print("qwen phase complete; elapsed_hours =", (time.time() - NOTEBOOK_START_TIME) / 3600)


In [ ]:
if RUN_INFERENCE:
    import base64
    import subprocess
    import sys
    from pathlib import Path

    wheel_payloads = {'adam_atan2_pytorch-0.2.4-py3-none-any.whl': '...', 'argdantic-1.3.3-py2.py3-none-any.whl': '...', 'coolname-2.2.0-py2.py3-none-any.whl': '...', 'hydra_core-1.3.2-py3-none-any.whl': '...'}
    wheel_dir = Path("/kaggle/working/trm_wheels")
    wheel_dir.mkdir()
    for name, payload in wheel_payloads.items():
        (wheel_dir / name).write_bytes(base64.b64decode(payload))

    subprocess.run([
        "uv", "pip", "install", "--system", "--no-index", "--no-deps",
        *[str(path) for path in sorted(wheel_dir.glob("*.whl"))],
    ], check=True)
    print("installed offline TRM wheels", sorted(wheel_payloads))


In [ ]:
if RUN_INFERENCE:
    import hashlib
    import os
    import shutil
    import subprocess
    import sys
    from pathlib import Path

    TRM_WORK = Path("/kaggle/working/trm64_phase")
    shutil.rmtree(TRM_WORK, ignore_errors=True)
    TRM_WORK.mkdir(parents=True)
    os.chdir(TRM_WORK)

    TRM_ROOT = Path("/kaggle/input/trm-code/20251211_trm_handover/TinyRecursiveModels")
    assert TRM_ROOT.is_dir(), TRM_ROOT
    observed_trm_hash = hashlib.sha256(
        (TRM_ROOT / "models/recursive_reasoning/trm.py").read_bytes()
    ).hexdigest()
    assert observed_trm_hash == "3454818b64a1abb45c062782d380aa4bb560e2cb35db9cc309cf367ca3ac262c"
    for name in ["models", "puzzle_dataset.py", "dataset", "utils", "evaluators", "assets", "config", "kaggle"]:
        os.symlink(TRM_ROOT / name, name)

    data_dir = Path("data1")
    data_dir.mkdir()
    shutil.copy2(TEST_PATH, data_dir / "arc-agi_test_challenges.json")
    subprocess.run([
        sys.executable, "-m", "dataset.build_arc_dataset",
        "--input-file-prefix", "./data1/arc-agi",
        "--output-dir", "data1/arc2test-aug-64",
        "--subsets", "test",
        "--test-set-name", "test",
        "--num-aug", str(TRM_AUGMENTATIONS),
    ], check=True)
    print("TRM dataset ready; elapsed_hours =", (time.time() - NOTEBOOK_START_TIME) / 3600)


In [ ]:
if RUN_INFERENCE:
    from pathlib import Path
    Path('eval-arc.py').write_text('\nfrom typing import Optional, Any, Sequence, List\nfrom dataclasses import dataclass\nimport os\nimport math\nimport yaml\nimport shutil\nimport copy\nimport time\n\nimport torch\nimport torch.distributed as dist\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nimport tqdm\n#import wandb\nimport coolname\nimport hydra\nimport pydantic\nfrom omegaconf import DictConfig\nfrom adam_atan2_pytorch import AdamAtan2\n\nfrom puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata\nfrom utils.functions import load_model_class, get_model_source_path\nfrom models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed\nfrom models.ema import EMAHelper\n\n\nclass LossConfig(pydantic.BaseModel):\n    model_config = pydantic.ConfigDict(extra=\'allow\')\n    name: str\n\n\nclass ArchConfig(pydantic.BaseModel):\n    model_config = pydantic.ConfigDict(extra=\'allow\')\n    name: str\n    loss: LossConfig\n\n\nclass EvaluatorConfig(pydantic.BaseModel):\n    model_config = pydantic.ConfigDict(extra="allow")\n    name: str\n\n\nclass PretrainConfig(pydantic.BaseModel):\n    # Config\n    arch: ArchConfig\n    # Data\n    data_paths: List[str]\n    data_paths_test: List[str] = []\n    # Evaluators\n    evaluators: List[EvaluatorConfig] = []\n\n    # Hyperparams\n    global_batch_size: int\n    epochs: int\n\n    lr: float\n    lr_min_ratio: float\n    lr_warmup_steps: int\n\n    weight_decay: float\n    beta1: float\n    beta2: float\n\n    # Puzzle embedding\n    puzzle_emb_lr: float\n    puzzle_emb_weight_decay: float\n\n    # Names\n    project_name: Optional[str] = None\n    run_name: Optional[str] = None\n    load_checkpoint: Optional[str] = None\n    checkpoint_path: Optional[str] = None\n\n    # Extras\n    seed: int = 0\n    checkpoint_every_eval: bool = False\n    eval_interval: Optional[int] = None\n    min_eval_interval: Optional[int] = 0 # when to start eval\n    eval_save_outputs: List[str] = []\n\n    ema: bool = False # use Exponential-Moving-Average\n    ema_rate: float = 0.999 # EMA-rate\n    freeze_weights: bool = False # If True, freeze weights and only learn the embeddings\n\n@dataclass\nclass TrainState:\n    model: nn.Module\n    optimizers: Sequence[torch.optim.Optimizer]\n    optimizer_lrs: Sequence[float]\n    carry: Any\n\n    step: int\n    total_steps: int\n\n\ndef create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):\n    dataset = PuzzleDataset(PuzzleDatasetConfig(\n        seed=config.seed,\n        dataset_paths=config.data_paths_test if len(config.data_paths_test)>0 and split=="test" else config.data_paths,\n        rank=rank,\n        num_replicas=world_size,\n        **kwargs\n    ), split=split)\n    dataloader = DataLoader(\n        dataset,\n        batch_size=None,\n        num_workers=1,\n        prefetch_factor=8,\n        pin_memory=True,\n        persistent_workers=True\n    )\n    return dataloader, dataset.metadata\n\n\ndef create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):\n    model_cfg = dict(\n        **config.arch.__pydantic_extra__,  # type: ignore\n        batch_size=config.global_batch_size // world_size,\n        vocab_size=train_metadata.vocab_size,\n        seq_len=train_metadata.seq_len,\n        num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,\n        causal=False  # Non-autoregressive\n    )\n\n    # Instantiate model with loss head\n    model_cls = load_model_class(config.arch.name)\n    loss_head_cls = load_model_class(config.arch.loss.name)\n\n    with torch.device("cuda"):\n        model: nn.Module = model_cls(model_cfg)\n        print(model)\n        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore\n        if "DISABLE_COMPILE" not in os.environ:\n            model = torch.compile(model)  # type: ignore\n\n        # Load checkpoint\n        if rank == 0:\n            load_checkpoint(model, config)\n\n        # Broadcast parameters from rank 0\n        if world_size > 1:\n            with torch.no_grad():\n                for param in list(model.parameters()) + list(model.buffers()):\n                    dist.broadcast(param, src=0)\n\n    # Optimizers and lr\n    if config.arch.puzzle_emb_ndim == 0:\n        optimizers = [\n            AdamAtan2(\n                model.parameters(),\n                lr=0.0001,  # Needs to be set by scheduler\n                weight_decay=config.weight_decay,\n                betas=(config.beta1, config.beta2)\n            )\n        ]\n        optimizer_lrs = [\n            config.lr\n        ]\n    elif config.freeze_weights:\n        optimizers = [\n            CastedSparseEmbeddingSignSGD_Distributed(\n                model.model.puzzle_emb.buffers(),  # type: ignore\n                lr=0,  # Needs to be set by scheduler\n                weight_decay=config.puzzle_emb_weight_decay,\n                world_size=world_size\n            )\n        ]\n        optimizer_lrs = [\n            config.puzzle_emb_lr\n        ]\n    else:\n        optimizers = [\n            CastedSparseEmbeddingSignSGD_Distributed(\n                model.model.puzzle_emb.buffers(),  # type: ignore\n                lr=0,  # Needs to be set by scheduler\n                weight_decay=config.puzzle_emb_weight_decay,\n                world_size=world_size\n            ),\n            AdamAtan2(\n                model.parameters(),\n                lr=0.0001,  # Needs to be set by scheduler\n                weight_decay=config.weight_decay,\n                betas=(config.beta1, config.beta2)\n            )\n        ]\n        optimizer_lrs = [\n            config.puzzle_emb_lr,\n            config.lr\n        ]\n\n    return model, optimizers, optimizer_lrs\n\ndef mix_weights_direct(device, alpha, net, nets):\n    sd = []\n    for i in range(len(nets)):\n        sd += [nets[i].state_dict()]\n    sd_alpha = {}\n    for k in sd[0].keys():\n        comb_net = alpha[0]*sd[0][k].to(device)\n        for i in range(1,len(nets)):\n            comb_net += alpha[i]*sd[i][k].to(device)\n        sd_alpha[k] =  comb_net\n    net.load_state_dict(sd_alpha)\n    return net\n\ndef cosine_schedule_with_warmup_lr_lambda(\n    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5\n):\n    if current_step < num_warmup_steps:\n        return base_lr * float(current_step) / float(max(1, num_warmup_steps))\n\n    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))\n    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))\n\n\ndef init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):\n    # Estimated total training steps\n    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)\n\n    # Model\n    model, optimizers, optimizer_lrs = create_model(config, train_metadata, rank=rank, world_size=world_size)\n\n    return TrainState(\n        step=0,\n        total_steps=total_steps,\n\n        model=model,\n        optimizers=optimizers,\n        optimizer_lrs=optimizer_lrs,\n        carry=None\n    )\n\n\ndef save_train_state(config: PretrainConfig, train_state: TrainState):\n    # FIXME: Only saved model.\n    if config.checkpoint_path is None:\n        return\n\n    os.makedirs(config.checkpoint_path, exist_ok=True)\n    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))\n\n\ndef load_checkpoint(model: nn.Module, config: PretrainConfig):\n    if config.load_checkpoint is not None:\n        print(f"Loading checkpoint {config.load_checkpoint}")\n\n        # Load state dict\n        state_dict = torch.load(config.load_checkpoint, map_location="cuda")\n\n        # Resize and reset puzzle emb if needed\n        puzzle_emb_name = "_orig_mod.model.inner.puzzle_emb.weights"\n        expected_shape: torch.Size = model.model.puzzle_emb.weights.shape  # type: ignore\n        if puzzle_emb_name in state_dict:\n            puzzle_emb = state_dict[puzzle_emb_name]\n            if puzzle_emb.shape != expected_shape:\n                print(f"Resetting puzzle embedding as shape is different. Found {puzzle_emb.shape}, Expected {expected_shape}")\n                # Re-initialize using mean\n                state_dict[puzzle_emb_name] = (\n                    torch.mean(puzzle_emb, dim=0, keepdim=True).expand(expected_shape).contiguous()\n                )\n        model.load_state_dict(state_dict, assign=True)\n\n\ndef compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):\n    return cosine_schedule_with_warmup_lr_lambda(\n        current_step=train_state.step,\n        base_lr=base_lr,\n        num_warmup_steps=round(config.lr_warmup_steps),\n        num_training_steps=train_state.total_steps,\n        min_ratio=config.lr_min_ratio\n    )\n\n\n\ndef create_evaluators(config: PretrainConfig, eval_metadata: PuzzleDatasetMetadata) -> List[Any]:\n    data_paths = config.data_paths_test if len(config.data_paths_test)>0 else config.data_paths\n    # Initialize evaluators\n    #print(\'evaluator\', data_paths, config.evaluators)\n    evaluators = []\n    for cfg in config.evaluators:\n        for data_path in data_paths:\n            klass = load_model_class(cfg.name, "evaluators.")\n            #print(\'klass\', klass)\n            cls = klass(\n                data_path=data_path, eval_metadata=eval_metadata, **cfg.__pydantic_extra__\n            )  # type: ignore\n            #print(\'cls\', cls)\n            evaluators.append(cls)\n\n    return evaluators\n\ndef train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):\n    train_state.step += 1\n    if train_state.step > train_state.total_steps:  # At most train_total_steps\n        return\n\n    # To device\n    batch = {k: v.cuda() for k, v in batch.items()}\n\n    # Init carry if it is None\n    if train_state.carry is None:\n        with torch.device("cuda"):\n            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore\n\n    # Forward\n    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])\n\n    ((1 / global_batch_size) * loss).backward()\n\n    # Allreduce\n    if world_size > 1:\n        for param in train_state.model.parameters():\n            if param.grad is not None:\n                dist.all_reduce(param.grad)\n            \n    # Apply optimizer\n    lr_this_step = None    \n    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):\n        lr_this_step = compute_lr(base_lr, config, train_state)\n\n        for param_group in optim.param_groups:\n            param_group[\'lr\'] = lr_this_step\n            \n        optim.step()\n        optim.zero_grad()\n\n    # Reduce metrics\n    if len(metrics):\n        assert not any(v.requires_grad for v in metrics.values())\n\n        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.\n        # Reduce and reconstruct\n        metric_values = torch.stack([metrics[k] for k in metric_keys])\n        if world_size > 1:\n            dist.reduce(metric_values, dst=0)\n\n        if rank == 0:\n            metric_values = metric_values.cpu().numpy()\n            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}\n            \n            # Postprocess\n            count = max(reduced_metrics["count"], 1)  # Avoid NaNs\n            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}\n\n            reduced_metrics["train/lr"] = lr_this_step\n            return reduced_metrics\n\ndef evaluate(\n    config: PretrainConfig,\n    train_state: TrainState,\n    eval_loader: torch.utils.data.DataLoader,\n    eval_metadata: PuzzleDatasetMetadata,\n    evaluators: List[Any],\n    rank: int,\n    world_size: int,\n    cpu_group: Optional[dist.ProcessGroup],\n):\n    reduced_metrics = None\n\n    with torch.inference_mode():\n        return_keys = set(config.eval_save_outputs)\n        for evaluator in evaluators:\n            evaluator.begin_eval()\n            return_keys.update(evaluator.required_outputs)\n\n        # Run evaluation\n        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}\n\n        save_preds = {}\n\n        metric_keys = []\n        metric_values = None\n\n        carry = None\n        processed_batches = 0\n        \n        for set_name, batch, global_batch_size in eval_loader:\n            processed_batches += 1\n            if rank == 0:\n                print(f"Processing batch {processed_batches}: {set_name}")\n            \n            # To device\n            batch = {k: v.cuda() for k, v in batch.items()}\n            with torch.device("cuda"):\n                carry = train_state.model.initial_carry(batch)  # type: ignore\n\n            # Forward\n            inference_steps = 0\n            while True:\n                carry, loss, metrics, preds, all_finish = train_state.model(\n                    carry=carry, batch=batch, return_keys=return_keys\n                )\n                inference_steps += 1\n\n                if all_finish:\n                    break\n\n            if rank == 0:\n                print(f"  Completed inference in {inference_steps} steps")\n\n            for collection in (batch, preds):\n                for k, v in collection.items():\n                    if k in config.eval_save_outputs:\n                        save_preds.setdefault(k, [])\n                        save_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory\n\n            for evaluator in evaluators:\n                evaluator.update_batch(batch, preds)\n\n            del carry, loss, preds, batch, all_finish\n\n            # Aggregate metrics\n            set_id = set_ids[set_name]\n\n            if metric_values is None:\n                metric_keys = list(\n                    sorted(metrics.keys())\n                )  # Sort keys to guarantee all processes use the same order.\n                metric_values = torch.zeros(\n                    (len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda"\n                )\n\n            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])\n\n            del metrics\n\n        # concatenate save preds\n        save_preds = {k: torch.cat(v, dim=0) for k, v in save_preds.items()}\n\n        # Save preds\n        if config.checkpoint_path is not None and len(save_preds):\n            # Each rank save predictions independently\n            os.makedirs(os.path.dirname(config.checkpoint_path), exist_ok=True)\n            torch.save(\n                save_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}")\n            )\n\n        del save_preds\n\n        # Reduce to rank 0\n        if metric_values is not None:\n            if world_size > 1:\n                dist.reduce(metric_values, dst=0)\n\n            if rank == 0:\n                reduced_metrics = metric_values.cpu().numpy()\n                reduced_metrics = {\n                    set_name: {\n                        metric_name: reduced_metrics[set_id, metric_id]\n                        for metric_id, metric_name in enumerate(metric_keys)\n                    }\n                    for set_id, set_name in enumerate(set_ids)\n                }\n\n                # Postprocess\n                for set_name, m in reduced_metrics.items():\n                    count = m.pop("count")\n                    reduced_metrics[set_name] = {k: v / count for k, v in m.items()}\n\n        # Run evaluators\n        if rank == 0:\n            print(f"\\nRunning {len(evaluators)} evaluator(s)...")\n            \n        for i, evaluator in enumerate(evaluators):\n            if rank == 0:\n                print(f"Running evaluator {i+1}/{len(evaluators)}: {evaluator.__class__.__name__}")\n                \n            # Path for saving\n            evaluator_save_path = None\n            if config.checkpoint_path is not None:\n                evaluator_save_path = os.path.join(\n                    config.checkpoint_path,\n                    f"evaluator_{evaluator.__class__.__name__}_step_{train_state.step}",\n                )\n                os.makedirs(evaluator_save_path, exist_ok=True)\n\n            # Run and log\n            metrics = evaluator.result(evaluator_save_path, rank=rank, world_size=world_size, group=cpu_group)\n            if rank == 0 and metrics is not None:\n                if reduced_metrics is None:\n                    reduced_metrics = {}\n\n                reduced_metrics.update(metrics)\n                print(f"  Completed {evaluator.__class__.__name__}")\n                \n        if rank == 0:\n            print("All evaluators completed!")\n\n    return reduced_metrics\n\ndef save_code_and_config(config: PretrainConfig):\n    if config.checkpoint_path is None:\n        return\n\n    os.makedirs(config.checkpoint_path, exist_ok=True)\n\n    # Copy code\n    code_list = [\n        get_model_source_path(config.arch.name),\n        get_model_source_path(config.arch.loss.name)\n    ]\n    for code_file in code_list:\n        if code_file is not None:\n            code_name = os.path.basename(code_file)\n\n            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))\n\n    # Dump config as yaml\n    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")\n    with open(config_file, "wt") as f:\n        yaml.dump(config.model_dump(), f)\n\n    # Log code\n    print(config.checkpoint_path)\n\n\ndef load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:\n    objects = [None]\n    if rank == 0:\n        config = PretrainConfig(**hydra_config)  # type: ignore\n\n        # Naming\n        if config.project_name is None:\n            config.project_name = f"{os.path.basename(config.data_paths[0]).capitalize()}-ACT-torch"\n        if config.run_name is None:\n            config.run_name = f"{config.arch.name.split(\'@\')[-1]} {coolname.generate_slug(2)}"\n        if config.checkpoint_path is None:\n            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)\n\n        objects = [config]\n\n    if world_size > 1:\n        dist.broadcast_object_list(objects, src=0)\n\n    return objects[0]  # type: ignore\n\n\n@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)\ndef launch(hydra_config: DictConfig):\n    RANK = 0\n    WORLD_SIZE = 1\n    CPU_PROCESS_GROUP = None\n\n    # Initialize distributed training if in distributed environment (e.g. torchrun)\n    if "LOCAL_RANK" in os.environ:\n        # Initialize distributed, default device and dtype\n        dist.init_process_group(backend="nccl")\n\n        RANK = dist.get_rank()\n        WORLD_SIZE = dist.get_world_size()\n\n        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))\n        \n        # CPU GLOO process group\n        CPU_PROCESS_GROUP = dist.new_group(backend="gloo")\n        assert (\n            dist.get_rank(CPU_PROCESS_GROUP) == RANK and dist.get_world_size(CPU_PROCESS_GROUP) == WORLD_SIZE\n        )\n\n    # Load sync\'ed config\n    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)\n\n    # Seed RNGs to ensure consistency\n    torch.random.manual_seed(config.seed + RANK)\n\n    # Dataset\n    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs\n    total_iters = config.epochs // train_epochs_per_iter\n\n    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."\n\n    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)\n    try:\n        eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)\n    except:\n        print("NO EVAL DATA FOUND")\n        eval_loader = eval_metadata = None\n\n    try:\n        evaluators = create_evaluators(config, eval_metadata)\n    except:\n        print("No evaluator found")\n        evaluators = []\n\n    # Train state\n    train_state = init_train_state(config, train_metadata, rank=RANK, world_size=WORLD_SIZE)\n\n    # Progress bar and logger\n    progress_bar = None\n    ema_helper = None\n    if RANK == 0:\n        progress_bar = tqdm.tqdm(total=train_state.total_steps)\n        print({"num_params": sum(x.numel() for x in train_state.model.parameters())})\n        save_code_and_config(config)\n    if config.ema:\n        print(\'Setup EMA\')\n        ema_helper = EMAHelper(mu=config.ema_rate)\n        ema_helper.register(train_state.model)\n\n    # Training Loop\n    for _iter_id in range(total_iters):\n        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")\n\n        ############ Train Iter\n        if RANK == 0:\n            print("TRAIN")\n        train_state.model.train()\n        for set_name, batch, global_batch_size in train_loader:\n            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)\n\n            if RANK == 0 and metrics is not None:\n                #print(metrics, train_state.step)\n                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore\n            if config.ema:\n                ema_helper.update(train_state.model)\n\n            deadline = float(os.environ.get("TRM_TRAIN_DEADLINE", "inf"))\n            stop = torch.tensor([int(time.time() >= deadline)], device="cuda")\n            if dist.is_initialized():\n                dist.all_reduce(stop, op=dist.ReduceOp.MAX)\n            if stop.item():\n                if RANK == 0:\n                    print(f"TRM training deadline reached at step {train_state.step}")\n                break\n\n        if _iter_id >= config.min_eval_interval:\n            ############ Evaluation\n            if RANK == 0:\n                print("EVALUATE")\n            if config.ema:\n                print("SWITCH TO EMA")\n                train_state_eval = copy.deepcopy(train_state)\n                train_state_eval.model = ema_helper.ema_copy(train_state_eval.model)\n            else:\n                train_state_eval = train_state\n            train_state_eval.model.eval()\n            metrics = evaluate(config, \n                train_state_eval, \n                eval_loader, \n                eval_metadata, \n                evaluators,\n                rank=RANK, \n                world_size=WORLD_SIZE,\n                cpu_group=CPU_PROCESS_GROUP)\n\n            if RANK == 0 and metrics is not None:\n                print(metrics, train_state.step)\n                \n            ############ Checkpointing\n            if RANK == 0:\n                print("SAVE CHECKPOINT")\n            if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):\n                save_train_state(config, train_state_eval)\n\n            if config.ema:\n                del train_state_eval\n\n    # finalize\n    if dist.is_initialized():\n        dist.destroy_process_group()\n\n\n\nif __name__ == "__main__":\n    launch()')
    print('Wrote eval-arc.py')


In [ ]:
if RUN_INFERENCE:
    import os
    import subprocess
    import time
    from pathlib import Path

    trm_deadline = NOTEBOOK_START_TIME + TRM_TRAIN_STOP_HOURS * 3600
    env = os.environ.copy()
    env["TRM_TRAIN_DEADLINE"] = str(trm_deadline)
    cmd = [
        "torchrun", "--standalone", "--nnodes=1", "--nproc-per-node", "4",
        "--rdzv_backend=c10d", "--rdzv_endpoint=localhost:0", "eval-arc.py",
        "arch=trm", "data_paths=[./data1/arc2test-aug-64]",
        "arch.L_layers=2", "arch.H_cycles=4", "arch.L_cycles=4",
        "arch.halt_max_steps=10", "freeze_weights=False",
        "+load_checkpoint=/kaggle/input/arc-prize-trm-031/step_220708",
        "+checkpoint_path=./eval_checkpoint", "eval_interval=4000", "epochs=4000",
        "global_batch_size=128", "ema=True", "lr_warmup_steps=200", "lr=0.0001",
    ]
    print("TRM seconds available for setup/training =", max(0, trm_deadline - time.time()))
    result = subprocess.run(cmd, env=env, check=False)
    print("TRM returncode =", result.returncode)


In [ ]:
import json
import shutil
from pathlib import Path

def valid_grid(grid):
    return (
        isinstance(grid, list) and 1 <= len(grid) <= 30
        and all(isinstance(row, list) and len(row) == len(grid[0]) for row in grid)
        and 1 <= len(grid[0]) <= 30
        and all(isinstance(cell, int) and 0 <= cell <= 9 for row in grid for cell in row)
    )

FINAL_SUBMISSION_PATH = Path("/kaggle/working/submission.json")
if not RUN_INFERENCE:
    shutil.copy2(QWEN_SUBMISSION_PATH, FINAL_SUBMISSION_PATH)
    print("Shortcut run: copied Qwen placeholder submission")
else:
    qwen_submission = json.loads(QWEN_SUBMISSION_PATH.read_text())
    submission_files = sorted(Path("eval_checkpoint").glob("evaluator_*/submission.json"))
    trm_submission = json.loads(submission_files[-1].read_text()) if submission_files else {}

    trm_used = 0
    fallback_used = 0
    for task, outputs in qwen_submission.items():
        trm_outputs = trm_submission.get(task, [])
        for output_index, qwen_output in enumerate(outputs):
            qwen_first = qwen_output["attempt_1"]
            qwen_second = qwen_output["attempt_2"]
            trm_first = None
            if output_index < len(trm_outputs):
                trm_first = trm_outputs[output_index].get("attempt_1")
            if valid_grid(trm_first) and trm_first != qwen_first:
                qwen_output["attempt_2"] = trm_first
                trm_used += 1
            else:
                qwen_output["attempt_2"] = qwen_second
                fallback_used += 1

    FINAL_SUBMISSION_PATH.write_text(json.dumps(qwen_submission))
    assert set(qwen_submission) == set(json.loads(Path(TEST_PATH).read_text()))
    print("TRM attempt-1 slots used =", trm_used)
    print("Qwen rank-2 fallbacks =", fallback_used)
print("final elapsed_hours =", (time.time() - NOTEBOOK_START_TIME) / 3600)
print("submission_path =", FINAL_SUBMISSION_PATH)
